In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import numpy as np
import torch

from src.ddm.ddm import DecisionModel, FreeParam, FixedParam

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

In [ ]:
COHERENCES     = np.array([-0.50, -0.18, -0.09, 0.0, 0.09, 0.18, 0.50]).round(2)
TRIALS_PER_COH = 150
MAX_DURATION     = 3.0
DT               = 0.001
CAF_WEIGHT       = 1
CAF_BINS         = 15
RT_VAR_WEIGHT    = 1

In [ ]:
def plot_mean_data(data):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    cohs_sorted = sorted(data["signed_coherence"].unique())


    def summarize(df):
        p_upper = []
        mean_rt_upper, sem_rt_upper = [], []
        mean_rt_lower, sem_rt_lower = [], []
        for c in cohs_sorted:
            subset = df[df["signed_coherence"] == c].dropna(subset=["rt", "choice"])
            p_upper.append((subset["choice"] == 1).mean() if len(subset) > 0 else np.nan)
            upper = subset[subset["choice"] == 1]["rt"]
            lower = subset[subset["choice"] == 0]["rt"]
            mean_rt_upper.append(upper.mean() if len(upper) > 0 else np.nan)
            sem_rt_upper.append(upper.sem() if len(upper) > 0 else np.nan)
            mean_rt_lower.append(lower.mean() if len(lower) > 0 else np.nan)
            sem_rt_lower.append(lower.sem() if len(lower) > 0 else np.nan)
        return np.array(p_upper), np.array(mean_rt_upper), np.array(sem_rt_upper), np.array(mean_rt_lower), np.array(sem_rt_lower)

    p_upper, mean_rt_upper, sem_rt_upper, mean_rt_lower, sem_rt_lower = summarize(data)

    # Psychometric function
    axes[0].plot(cohs_sorted, p_upper, "o-", color="steelblue")
    axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    axes[0].axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    axes[0].set_xlabel("Coherence")
    axes[0].set_ylabel("P(upper boundary)")
    axes[0].set_title("Psychometric Function")
    axes[0].set_ylim(0, 1)

    # Chronometric function
    mean_rt  = []
    sem_rt   = []
    for coh in cohs_sorted:
        subset = data[data["signed_coherence"] == coh].dropna()
        mean_rt.append(subset["rt"].mean())
        sem_rt.append(subset["rt"].sem())

    axes[1].errorbar(cohs_sorted, mean_rt, yerr=sem_rt, fmt="o-", color="tomato", capsize=4)
    axes[1].axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    axes[1].set_xlabel("Coherence")
    axes[1].set_ylabel("Mean RT (s)")
    axes[1].set_title("Chronometric Function")

    #
    # --- Chronometric ---

    ax2 = axes[2]
    ax2.errorbar(cohs_sorted, mean_rt_upper, yerr=sem_rt_upper, fmt="o-", color="steelblue", capsize=4, label="choice 1")
    ax2.errorbar(cohs_sorted, mean_rt_lower, yerr=sem_rt_lower, fmt="o-", color="tomato", capsize=4, label="choice 0")
    ax2.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax2.set(xlabel="Coherence", ylabel="Mean RT (s)", title="Chronometric")
    ax2.legend()

    # ax2.sharey(axes[1])  # share y-axis with the middle plot|
    axes[1].sharey(ax2)  # share y-axis with the right plot

    plt.tight_layout()
    plt.show()

# for each coherence plot histogram of RTs for choice 0 and choice 1
def plot_raw_data(data):

    fig, axes = plt.subplots(len(data["signed_coherence"].unique()), 1, figsize=(8, 3 * len(data["signed_coherence"].unique())))

    for ax, coh in zip(axes, sorted(data["signed_coherence"].unique())):
        subset = data[data["signed_coherence"] == coh]
        ax.hist(subset[subset["choice"] == 0]["rt"], bins=20, alpha=0.5, label="choice 0", color="steelblue")
        ax.hist(subset[subset["choice"] == 1]["rt"], bins=20, alpha=0.5, label="choice 1", color="tomato")
        ax.set_title(f"Coherence: {coh:+.2f}")
        ax.set_xlabel("RT (s)")
        ax.set_ylabel("Count")
        ax.set_ylim(0, data["rt"].max() + 0.5)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
fixed_params = {
    "dt": FixedParam(0.001),
    "variance": FixedParam(1.0),
    "sz": FixedParam(0.0),
    "sv": FixedParam(0.0),
}

free_params = {
    "ndt": FreeParam(0.1, 1),
    "a": FreeParam(0.8, 6.0),
    "z": FreeParam(0.1, 0.9),
    "drift_gain": FreeParam(1.0, 10.0),
    "drift_offset": FreeParam(-5.0, 5.0),
    # "variance": FreeParam(0.5, 2.0),
    "leak_rate": FreeParam(0.0, 0.3),
    "time_constant": FreeParam(-2.0, 2.0),
}

likelihood_params = {
    "chrono_weight": 1.5,
    "caf_weight":    1.5,
    "rt_var_weight": 0.0,
    "caf_bins":      5,
}
class DDMModel(DecisionModel):

    def __init__(self, fixed_params, free_params, likelihood_params=None):

        super().__init__(free_params=free_params, fixed_params=fixed_params, device=device, likelihood_params=likelihood_params)

    def _objective_function(self, values, data, stimulus, n_reps, l1_weight):
        try:
            params = self._build_params(values)
        except ValueError:
            return 1e6
        result = self._simulate_condition(stimulus, params, n_reps)
        if result is None:
            return 1e6
        return self.likelihood_calc.compute_nll(
            result["rt"], result["choice"],
            np.asarray(data["rt"]), np.asarray(data["choice"]),
            result["signed_coherence"], np.asarray(data["signed_coherence"]),
        )

ddm_model = DDMModel(
    fixed_params=fixed_params,
    free_params=free_params,
    likelihood_params=likelihood_params
)

# Generate Data

In [ ]:
def make_stimulus() -> np.ndarray:
    n_tp = int(MAX_DURATION / DT)
    coh  = np.repeat(COHERENCES, TRIALS_PER_COH)
    return np.tile(coh.reshape(-1, 1), (1, n_tp)).astype(np.float32)

def generate_data(params: dict = None, seed: int = None):
    """Like generate_data() but uses the src/ddm/ddm.py dict-based API."""
    if params is None:
        params = dict(DEFAULT_PARAMS)

    stimulus   = make_stimulus()
    sim_data = ddm_model.simulate(stimulus=stimulus, params=params, seed=seed)

    return sim_data

In [ ]:
# calculate likelihood for original parameters (before fitting) for comparison
original_params = {
    "ndt": 0.25,
    "a": 1.0,
    "z": 0.5,
    "drift_gain": 8.0,
    "drift_offset": 0.0,
    "variance": 1.0,
    "leak_rate": 0.1,
    "time_constant": 0,#0.5,
    "dt": 0.001,

}

stimulus = make_stimulus()

data = generate_data(params=original_params, seed=42)

# count number of 0, 1 and NaN in each coherence bin
summary = data.groupby("signed_coherence")["choice"].agg(
    n_trials="count",
    n_choice_0=lambda x: np.sum(x == 0),
    n_choice_1=lambda x: np.sum(x == 1),
    n_nan=lambda x: np.sum(np.isnan(x)),
).reset_index()

likelihood = ddm_model.likelihood_calc.compute_nll(
    data["rt"], data["choice"],
    data["rt"], data["choice"],
    data["signed_coherence"], data["signed_coherence"],
)

print(f"Data likelihood under original parameters: {likelihood:.1f}")


print(summary)

plot_mean_data(data)
# plot_raw_data(data)

# Recovery Trial

### DDM Fitting

In [ ]:
result = ddm_model.fit(
    data={"rt": data["rt"], "choice": data["choice"], "signed_coherence": data["signed_coherence"]},
    stimulus=stimulus,
    max_iterations=900,
    n_reps=15,
    l1_weight=0,
)

# plot model simulations

In [ ]:
def plot_ddm_fit(data, sim, coherences=None, title="DDM Fit"):
    """
    Overlay observed data (dots) and model predictions (lines).

    data : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    sim  : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    """
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    if not isinstance(sim, pd.DataFrame):
        sim = pd.DataFrame(sim)

    cohs = sorted(coherences if coherences is not None else data["signed_coherence"].unique())

    def summarize(df):
        p_upper, mean_rt_upper, mean_rt_lower = [], [], []
        for c in cohs:
            subset = df[df["signed_coherence"] == c].dropna(subset=["rt", "choice"])
            p_upper.append((subset["choice"] == 1).mean() if len(subset) > 0 else np.nan)
            upper = subset[subset["choice"] == 1]["rt"]
            lower = subset[subset["choice"] == 0]["rt"]
            mean_rt_upper.append(upper.mean() if len(upper) > 0 else np.nan)
            mean_rt_lower.append(lower.mean() if len(lower) > 0 else np.nan)
        return np.array(p_upper), np.array(mean_rt_upper), np.array(mean_rt_lower)

    d_p, d_rt1, d_rt0 = summarize(data)
    s_p, s_rt1, s_rt0 = summarize(sim)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    # --- Psychometric ---
    ax1.plot(cohs, s_p, color="steelblue", label="model")
    ax1.scatter(cohs, d_p, color="steelblue", zorder=5, label="data")
    ax1.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax1.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax1.set(xlabel="Coherence", ylabel="P(upper boundary)", title="Psychometric", ylim=(0, 1))
    ax1.legend()

    # --- Chronometric ---
    ax2.plot(cohs, s_rt1, color="steelblue", label="model upper")
    ax2.plot(cohs, s_rt0, color="tomato",    label="model lower")
    ax2.scatter(cohs, d_rt1, color="steelblue", zorder=5, label="data upper")
    ax2.scatter(cohs, d_rt0, color="tomato",    zorder=5, label="data lower")
    ax2.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax2.set(xlabel="Coherence", ylabel="Mean RT (s)", title="Chronometric")
    ax2.legend()

    plt.tight_layout()
    return fig

In [ ]:
# save model results
with open("model_results.pkl", "wb") as f:
    torch.save(result, f)

## Likelihood Calculator

In [ ]:
def model_comparison():
    fitted_params = result["parameters"]
    sim_data = ddm_model.simulate(stimulus, params=fitted_params)
    return sim_data

In [ ]:
with open("model_results.pkl", "rb") as f:
    result = torch.load(f)

In [ ]:
fitted_llh, origin_param_llh = [], []

sim_data_original  = ddm_model.simulate(
    stimulus, params=result["parameters"]
)

for i in range(1):
    sim_data_cuda = model_comparison()

    llh_cuda = ddm_model.likelihood_calc.compute_nll(
        sim_data_cuda["rt"].values, sim_data_cuda["choice"].values,
        data["rt"].values, data["choice"].values,
        sim_data_cuda["signed_coherence"].values, data["signed_coherence"].values,
    )
    llh_original_param = ddm_model.likelihood_calc.compute_nll(
        sim_data_original["rt"].values, sim_data_original["choice"].values,
        data["rt"].values, data["choice"].values,
        sim_data_original["signed_coherence"].values, data["signed_coherence"].values,
    )

    fitted_llh.append(llh_cuda)
    origin_param_llh.append(llh_original_param)

llh_data = ddm_model.likelihood_calc.compute_nll(
    data["rt"].values, data["choice"].values,
    data["rt"].values, data["choice"].values,
    data["signed_coherence"].values, data["signed_coherence"].values,
)

print(f"Data vs itself:   {llh_data:.1f}  (should be 0)")
print(f"Original params:  {np.mean(origin_param_llh):.1f} ± {np.std(origin_param_llh):.1f}")
print(f"Fitted params:    {np.mean(fitted_llh):.1f} ± {np.std(fitted_llh):.1f}  (should be ≤ original)")

In [ ]:
def plot_ddm_fit(data, sim, coherences=None, title="DDM Fit"):
    """
    Overlay observed data (dots) and model predictions (lines).

    data : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    sim  : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    """
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    if not isinstance(sim, pd.DataFrame):
        sim = pd.DataFrame(sim)

    data = data.copy()
    sim  = sim.copy()
    data["signed_coherence"] = data["signed_coherence"].round(2)
    sim["signed_coherence"]  = sim["signed_coherence"].round(2)

    cohs = sorted(coherences if coherences is not None else data["signed_coherence"].unique())

    def summarize(df):
        p_upper, mean_rt_upper, mean_rt_lower = [], [], []
        for c in cohs:
            subset = df[df["signed_coherence"] == c].dropna(subset=["rt", "choice"])
            p_upper.append((subset["choice"] == 1).mean() if len(subset) > 0 else np.nan)
            upper = subset[subset["choice"] == 1]["rt"]
            lower = subset[subset["choice"] == 0]["rt"]
            mean_rt_upper.append(upper.mean() if len(upper) > 0 else np.nan)
            mean_rt_lower.append(lower.mean() if len(lower) > 0 else np.nan)
        return np.array(p_upper), np.array(mean_rt_upper), np.array(mean_rt_lower)

    d_p, d_rt1, d_rt0 = summarize(data)
    s_p, s_rt1, s_rt0 = summarize(sim)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    # --- Psychometric ---
    ax1.plot(cohs, s_p, color="steelblue", label="model")
    ax1.scatter(cohs, d_p, color="steelblue", zorder=5, label="data")
    ax1.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax1.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax1.set(xlabel="Coherence", ylabel="P(upper boundary)", title="Psychometric", ylim=(0, 1))
    ax1.legend()

    # --- Chronometric ---
    ax2.plot(cohs, s_rt1, color="steelblue", label="model upper")
    ax2.plot(cohs, s_rt0, color="tomato",    label="model lower")
    ax2.scatter(cohs, d_rt1, color="steelblue", zorder=5, label="data upper")
    ax2.scatter(cohs, d_rt0, color="tomato",    zorder=5, label="data lower")
    ax2.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax2.set(xlabel="Coherence", ylabel="Mean RT (s)", title="Chronometric")
    ax2.legend()

    plt.tight_layout()
    # return fig

In [ ]:
plot_ddm_fit(sim_data_cuda, data, title="CUDA Fit vs Data")

# Hyperparameter Tuning

In [ ]:
import os

cwd = os.getcwd()
if cwd == "/src/notebooks":
    os.chdir("..")

from scripts.ddm.hyperparameter_tuning import run_tuning, plot_best_fit, build_param_grid
from pathlib import Path

# Narrow grid first — expand once you see which region works best.
# ~36 configs × ~5 min each ≈ 3 h on GPU; reduce max_iter to 150 for a quick scan.
TUNING_SAVE = Path("tuning_cache.pkl")

results_df = run_tuning(
    data=data,           # use the synthetic data already in memory
    stimulus=stimulus,   # same stimulus
    chrono_weights=[0.5, 1.0, 2.0, 4.0],
    caf_weights=[0.5, 1.0, 2.0],
    rt_var_weights=[0.0, 0.5, 1.0],
    n_reps=5,            # reduced for speed; use 15 for the final fit
    max_iter=300,        # reduced for speed; use 1000 for the final fit
    save_path=TUNING_SAVE,
    resume=True,         # safe to re-run: skips already-completed configs
)

results_df.head(10)

In [ ]:
import os

cwd = os.getcwd()
if cwd == "/src/notebooks":
    os.chdir("..")

from scripts.ddm.hyperparameter_tuning import run_tuning, plot_best_fit, build_param_grid
from pathlib import Path

# Narrow grid first — expand once you see which region works best.
# ~36 configs × ~5 min each ≈ 3 h on GPU; reduce max_iter to 150 for a quick scan.
TUNING_SAVE = Path("tuning_cache2.pkl")

results_df = run_tuning(
    data=data,           # use the synthetic data already in memory
    stimulus=stimulus,   # same stimulus
    chrono_weights=[1.0, 1.5, 2.0, 2.5],
    caf_weights=[1.5, 2.0, 2.5, 3.0],
    rt_var_weights=[0.0],
    n_reps=5,            # reduced for speed; use 15 for the final fit
    max_iter=300,        # reduced for speed; use 1000 for the final fit
    save_path=TUNING_SAVE,
    resume=True,         # safe to re-run: skips already-completed configs
)

results_df.head(10)

In [ ]:
import pickle
import os

cwd = os.getcwd()
if cwd == "/src/notebooks":
    os.chdir("..")

with open("tuning_cache2.pkl", "rb") as f:
    results_df = pickle.load(f)

In [ ]:
import pandas as pd

results = pd.DataFrame(results_df)

In [ ]:
results

In [ ]:
print("\n=== Top 5 configs ===")
cols = ["label", "composite", "psychometric", "chrono_upper", "chrono_lower", "nll"]
print(results_df[cols].head().to_string(index=False))